# Jour 1 -- Pandas -- SOLUTIONS (pédagogique)

Chaque exercice présente : la solution ; **POURQUOI CETTE APPROCHE** (+
alternatives) ; **PIÈGES / ERREURS FRÉQUENTES** ; **SCHÉMA À RECONNAÎtre**.


In [ ]:
# ---------------------------------------------------------------------------
# Run this cell first. Path is relative to this notebook's folder.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

DATA = "../data"
drivers = pd.read_csv(f"{DATA}/drivers.csv")
print(drivers.shape)

---
## Exercice 1 -- `read_csv`, `head`, `shape`, `columns`

### Solution


In [ ]:
display(drivers.head(5))
print(drivers.shape)
print(drivers.columns.tolist())


**POURQUOI.** `read_csv` est déjà fait dans la cellule de départ -- sur
CodeSignal, la donnée est presque toujours pré-chargée ou le chemin est
donné dans l'énoncé ; ne perdez pas de temps à deviner un chemin relatif.
`head(5)` répond à *"à quoi ressemble une ligne ?"*, `shape` à *"combien de
données ?"*, `columns` à *"quels noms exacts dois-je utiliser ensuite ?"`
(une faute de frappe dans un nom de colonne est l'erreur n°1 en examen
chronométré).

**SCHÉMA À RECONNAÎtre.** *"Nouveau fichier"* -> toujours ces trois lignes
avant d'écrire la moindre logique.


---
## Exercice 2 -- `info`

### Solution


In [ ]:
drivers.info()
# 5 colonnes ont au moins une valeur manquante : age, second_language,
# rating, city, years_active, status (soit 6 sur 7 -- seule driver_id est complète)


**POURQUOI.** `info()` donne en une seule commande ce qu'il faudrait
sinon trois lignes pour obtenir : dtype par colonne, nombre de non-nuls,
mémoire. Le signal le plus important à lire : une colonne qui **devrait**
être numérique mais apparaît en `object` cache presque toujours des
valeurs texte parasites.

**PIÈGE FRÉQUENT.** Confondre `info()` (dtypes + non-nuls, imprime
directement, retourne `None`) et `describe()` (statistiques, uniquement
colonnes numériques par défaut). Les deux se complètent, ne remplacent pas
l'un l'autre.

**SCHÉMA À RECONNAÎtre.** *"Diagnostiquer un dataset inconnu"* ->
`shape` + `head` + `info` + `isna().sum()` en moins de 30 secondes, avant
toute autre opération.


---
## Exercice 3 -- `isna`, `notna`

### Solution


In [ ]:
print(drivers.isna().sum())
print((drivers.isna().mean() * 100).round(1))

valid_rating = drivers[drivers["rating"].notna()]
assert valid_rating["rating"].isna().sum() == 0
print(valid_rating.shape)


**POURQUOI `mean()` POUR UN POURCENTAGE.** `isna()` produit une matrice
de `True`/`False` (1/0). La **moyenne** d'une colonne booléenne est
exactement sa **proportion de `True`** -- c'est le raccourci à connaître par
cœur : `bool.mean()` = taux. Multiplier par 100 donne le pourcentage.

**`notna()` vs `~isna()`.** Strictement équivalents ; `notna()` est plus
lisible et évite d'oublier le `~`.

**PIÈGE FRÉQUENT.** Utiliser `drivers.dropna()` sans `subset=` quand on ne
veut filtrer que sur **une** colonne -- ça supprime aussi les lignes ayant
un `NaN` ailleurs, un périmètre bien plus large que demandé.

**SCHÉMA À RECONNAÎtre.** *"Taux / pourcentage de condition"* ->
`condition_booleenne.mean()`. Ce réflexe revient sans cesse (valeurs
manquantes, taux de succès, part de telle catégorie...).


---
## Exercice 4 -- `loc`, `iloc`

### Solution


In [ ]:
subset_loc = drivers.loc[:, ["driver_id", "age", "rating"]]
subset_iloc = drivers.iloc[:10, :3]
mid_cities = drivers.loc[5:10, "city"]

print(subset_loc.shape, subset_iloc.shape, mid_cities.shape)


**`loc` = étiquettes, `iloc` = positions.** C'est la seule chose à
retenir. `drivers.loc[5:10]` renvoie **6** lignes (5,6,7,8,9,10 -- borné à
droite inclus) alors que `drivers.iloc[5:10]` en renvoie **5** (indexation
Python classique, 10 exclu). Cette différence est un classique de piège en
QCM/assessment.

**QUAND UTILISER QUOI.** `.loc` dès que vous raisonnez par **nom** de
colonne ou que l'index n'est pas un simple `RangeIndex` (par exemple après
un filtrage, où les positions ne correspondent plus aux étiquettes). `.iloc`
uniquement quand vous raisonnez en **position pure** ("les 10 premières
lignes", indépendamment de ce que vaut l'index).

**PIÈGE FRÉQUENT.** Enchaîner `df[df.rating > 4].iloc[3]` en pensant obtenir
la ligne d'index `3` -- non, c'est la 4e ligne du résultat filtré (l'index
d'origine n'est pas remis à zéro). Pour viser une étiquette précise après
filtrage, utilisez `.loc`.

**SCHÉMA À RECONNAÎtre.** *"sélectionner des colonnes par nom"* -> `.loc`;
*"les N premières lignes/colonnes"* -> `.iloc`.


---
## Exercice 5 -- Filtrage booléen et `filter`

### Solution


In [ ]:
top_rated = drivers[drivers["rating"] > 4.5]
suspicious_age = drivers[drivers["age"].isna() | (drivers["age"] > 100)]
rat_cols = drivers.filter(like="rat")

print(top_rated.shape, suspicious_age.shape)
print(rat_cols.columns.tolist())


**POURQUOI `&` / `|` ET DES PARENTHèSES.** En Python, `and`/`or` appellent
`bool()` sur l'objet entier -- indéfini pour une Series de plusieurs
valeurs (`ValueError: truth value is ambiguous`). Pandas fournit les
opérateurs bit à bit vectorisés `&`, `|`, `~`, qui comparent élément par
élément -- mais ils ont une priorité plus haute que `>`/`<`, d'où
l'obligation de parenthéser chaque condition : `(df.a > 1) & (df.b < 2)`.

**`DataFrame.filter` sélectionne des COLONNES (ou lignes) par NOM, pas par
valeur.** Ne pas confondre avec un filtrage de lignes par condition -- c'est
le piège principal de cet exercice. `like=` fait une recherche de
sous-chaîne ; `regex=` accepte une expression régulière ; `items=` une
liste exacte.

**PIÈGE FRÉQUENT.** Écrire `df[df.age.isna() or df.age > 100]` -> crash
immédiat. Ou oublier les parenthèses : `df.age.isna() | df.age > 100` est
interprété comme `df.age.isna() | (df.age > 100)` grâce à la précédence
réelle ici, mais dès qu'un `&`/`|` côtoie un `>`/`<` sans parenthèses sur
des cas plus complexes, le résultat devient imprévisible -- prenez
l'habitude de toujours parenthéser.

**SCHÉMA À RECONNAÎtre.** *"lignes qui vérifient une condition"* ->
masque booléen `df[(cond1) & cond2]` ; *"colonnes dont le nom contient X"*
-> `df.filter(like="X")`.


---
## Exercice 6 -- `isin`

### Solution


In [ ]:
main_cities = drivers[drivers["city"].isin(["Paris", "Lyon", "Nice"])]
other_cities = drivers[~drivers["city"].isin(["Paris", "Lyon", "Nice"])]

assert len(main_cities) + len(other_cities) == len(drivers)
print(len(main_cities), len(other_cities))


**POURQUOI `isin` PLUTÔT QU'UNE CHAÎne DE `|`.** `df.city.isin([a, b, c])`
est à la fois plus court et plus rapide que
`(df.city == a) | (df.city == b) | (df.city == c)` -- surtout lisible dès
qu'on dépasse 2 valeurs. C'est LE réflexe pour "appartient à une liste".

**`~` POUR L'INVERSE.** `~drivers.city.isin([...])` se lit "n'est pas dans".
Attention : les valeurs `NaN` de `city` ressortent comme `True` dans
`~isin` (car `NaN` n'est jamais dans la liste) -- vérifiez si l'énoncé
attend que les manquants soient inclus ou exclus du résultat.

**PIÈGE FRÉQUENT.** Comme ici la casse n'a pas été normalisée, `"paris"`
et `"PARIS"` **ne matchent pas** `"Paris"` dans `isin` -- `main_cities` ne
contient donc pas tous les chauffeurs parisiens. C'est volontaire dans ce
dataset pour bien retenir : `isin` compare des valeurs **exactes**, jamais
une variante de casse. Normaliser (`str.strip().str.title()`) est un
réflexe de Jour 2 (manipulation de texte), pas de Jour 1.

**SCHÉMA À RECONNAÎtre.** *"appartient à un ensemble de valeurs"* ->
`df[col].isin([...])` ; *"n'appartient pas"* -> préfixer par `~`.


---
## Exercice 7 -- `fillna`, `dropna`

### Solution


In [ ]:
d = drivers.copy()
d["rating"] = d["rating"].fillna(d["rating"].mean())
d["years_active"] = d["years_active"].fillna(0)
d["city"] = d["city"].fillna("Unknown")
d = d.dropna(subset=["driver_id", "second_language"])

print(d[["rating", "years_active", "city", "second_language"]].isna().sum())


**POURQUOI RÉAFFECTER (`d["col"] = ...fillna(...)`) PLUTÔT QUE
`inplace=True`.** `inplace=True` modifie l'objet en place mais renvoie
`None` -- si vous l'utilisez en chaîne ou l'assignez, vous perdez le
résultat. La réaffectation explicite fonctionne toujours et rend le code
lisible. C'est aussi la pratique recommandée par pandas depuis plusieurs
versions (les futures versions renforcent le comportement "copy-on-write").

**CHOIX DE LA VALEUR DE REMPLISSAGE -- ça dépend du sens métier, pas d'une
règle unique.**
* `rating` : numérique, distribution à peu près symétrique -> la
  **moyenne** est défendable (la **médiane** le serait tout autant en
  présence d'outliers -- à justifier à l'oral selon la distribution
  observée).
* `years_active` manquant -> on suppose ici l'absence d'info comme "pas
  encore démarré" -> `0` est un choix métier explicite, pas une moyenne
  statistique.
* `city` manquant -> aucune valeur numérique n'a de sens -> catégorie
  explicite `"Unknown"`, jamais inventée au hasard.

**`dropna(subset=[...])` PLUTÔT QUE `dropna()` SEUL.** `dropna()` sans
argument supprime une ligne dès qu'**une seule** colonne quelconque est
manquante -- sur ce dataset, ça viderait quasiment toute la table. Toujours
restreindre `subset=` aux colonnes réellement concernées par l'énoncé.

**PIÈGE FRÉQUENT.** Appeler `fillna` sur tout le DataFrame
(`drivers.fillna(drivers.mean())`) -- échoue ou produit des valeurs
absurdes sur les colonnes texte (`city`, `status`). Toujours cibler la
colonne.

**SCHÉMA À RECONNAÎtre.** *"remplir les manquants"* -> une valeur de
remplissage **par colonne**, justifiée par le sens de la colonne, jamais un
remplissage global uniforme.


---
## Exercice 8 -- `drop_duplicates`

### Solution


In [ ]:
n_exact_dup = drivers.duplicated().sum()
no_exact_dup = drivers.drop_duplicates()

drivers_unique = no_exact_dup.drop_duplicates(subset=["driver_id"], keep="last")

assert drivers_unique["driver_id"].is_unique
print(n_exact_dup, no_exact_dup.shape, drivers_unique.shape)


**DEUX NIVEAUX DE DOUBLON, DEUX TRAITEMENTS.**
* **Doublon exact** (`drop_duplicates()` sans argument) : toutes les
  colonnes sont identiques -> toujours sûr à supprimer, aucune perte
  d'information.
* **Doublon de clé métier** (`drop_duplicates(subset=["driver_id"])`) :
  l'identifiant se répète mais le contenu diffère (ici `rating` légèrement
  changé) -> il faut choisir une règle de survivant (`keep="first"`,
  `"last"`, ou une logique plus fine hors périmètre Jour 1).

**POURQUOI TRAITER LES DOUBLONS EXACTS D'ABORD.** Une fois les copies
identiques éliminées, l'étape `subset=["driver_id"]` ne compare plus que
des lignes **réellement différentes** -- le choix `keep=` devient
significatif au lieu de trancher entre deux copies parfaitement
équivalentes.

**PIÈGE FRÉQUENT.** Appliquer directement
`drivers.drop_duplicates(subset=["driver_id"])` sans avoir retiré les
doublons exacts au préalable -- le résultat final peut être correct par
chance ici, mais le raisonnement est incomplet et cassera sur un dataset où
les deux problèmes se combinent différemment.

**SCHÉMA À RECONNAÎtre.** *"une seule ligne par identifiant"* ->
`drop_duplicates(subset=["id"], keep=...)` puis toujours **vérifier** avec
`assert df["id"].is_unique` -- ne jamais supposer que ça a marché.


---
## Exercice 9 -- `sort_values`

### Solution


In [ ]:
by_rating = drivers.sort_values("rating", ascending=False)
display(by_rating.head())

by_city_age = drivers.sort_values(["city", "age"], ascending=[True, False])
by_city_age.head()


**TRI MULTI-CLéS.** Passer une **liste** de colonnes à `sort_values`
trie d'abord par la première clé, puis, à égalité, par la seconde. Le
paramètre `ascending` accepte aussi une liste, une valeur booléenne par
colonne -- ici croissant sur `city`, décroissant sur `age`.

**OÙ VONT LES `NaN` ?** Par défaut (`na_position="last"`), les valeurs
manquantes de la colonne de tri finissent toujours en bas, quel que soit
`ascending`. Si l'énoncé demande explicitement de les voir en premier,
`na_position="first"`.

**PIÈGE FRÉQUENT.** Oublier que `sort_values` renvoie un **nouveau**
DataFrame (pas de tri en place par défaut) -- toujours réaffecter le
résultat si vous voulez le conserver.

**SCHÉMA À RECONNAÎtre.** *"classer / top N / pire N"* ->
`sort_values(col, ascending=...)` (éventuellement suivi de `.head(N)`).


---
## Exercice 10 -- `value_counts`

### Solution


In [ ]:
counts = drivers["second_language"].value_counts(dropna=False)
pct = drivers["second_language"].value_counts(dropna=False, normalize=True) * 100

top_city = drivers["city"].value_counts().idxmax()

print(counts)
print(pct.round(1))
print("ville la plus frequente :", top_city)


**`value_counts()` COMPTE LES VALEURS DISTINCTES ET TRIE DÉJÀ PAR
FRÉQUENCE DÉCROISSANTE** -- pas besoin d'un `sort_values` derrière.

**`dropna=False`.** Par défaut, `value_counts()` **ignore** les `NaN` --
un piège classique si l'énoncé demande le décompte "y compris les
manquants". Le paramètre `dropna=False` les fait apparaître comme une
catégorie `NaN` à part entière.

**`normalize=True` POUR UN POURCENTAGE.** Évite de diviser manuellement par
`len(df)` -- une ligne, pas d'erreur possible sur le dénominateur (ce
denominateur exclut cependant les NaN si `dropna=True`, à garder en tête).

**`idxmax()` PLUTÔT QUE `.index[0]` APRèS TRI.** `value_counts().idxmax()`
donne directement le **label** de la valeur la plus fréquente en une seule
expression -- pas besoin de trier puis d'indexer.

**SCHÉMA À RECONNAÎtre.** *"distribution d'une colonne catégorielle"* ->
`value_counts(dropna=False, normalize=True)` ; *"la valeur la plus
fréquente"* -> `value_counts().idxmax()`.


---
# RÉCAPITULATIF -- LES 16 RÉFLEXES DE LA SESSION

| Besoin | Instruction |
|---|---|
| charger un CSV | `pd.read_csv(path)` |
| aperçu rapide | `df.head(n)` |
| diagnostic complet | `df.info()` |
| dimensions | `df.shape` |
| noms de colonnes | `df.columns` |
| sélection par nom | `df.loc[rows, cols]` |
| sélection par position | `df.iloc[rows, cols]` |
| colonnes par motif de nom | `df.filter(like="x")` |
| appartenance à une liste | `df[col].isin([...])` |
| valeurs manquantes | `df.isna()` / `.notna()` |
| remplir les manquants | `s.fillna(valeur)` |
| supprimer les manquants | `df.dropna(subset=[...])` |
| supprimer les doublons | `df.drop_duplicates(subset=[...], keep=...)` |
| trier | `df.sort_values(col, ascending=...)` |
| compter les valeurs distinctes | `s.value_counts(dropna=False, normalize=...)` |
| taux/pourcentage d'une condition | `condition_booleenne.mean()` |

Si vous avez mis plus de 10 minutes pour tout le notebook, refaites-le
demain matin à froid, sans relire les solutions, avant de passer au
Jour 2.
